# Phase 3 — Analyze
## 02 —Inventory / Stock Analysis

## Objective

Analyse the governed stock-on-hand dataset to understand the current inventory
position, stock availability, inventory concentration, and alignment between
stock holdings and observed product demand.

This notebook uses the Phase 2 governed `fact_stock.csv` dataset.

Raw stock data is not cleaned or transformed again in Phase 3. Source-level
files are consulted only when an analytical result requires investigation.

In [1]:
# ============================================================
# LOAD GOVERNED STOCK DATA
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

stock_path = Path("../../data/processed/fact_stock.csv")

fact_stock = pd.read_csv(stock_path)

print("GOVERNED STOCK DATA")
print("=" * 70)

print(f"Rows    : {len(fact_stock):,}")
print(f"Columns : {fact_stock.shape[1]:,}")

print("\nCOLUMNS")
print("=" * 70)

for col in fact_stock.columns:
    print(col)

print("\nSAMPLE")
display(fact_stock.head(10))

GOVERNED STOCK DATA
Rows    : 2,681
Columns : 7

COLUMNS
Model
Store
Category
Description
Quantity
Stock_Status
Outstanding_Order_Qty

SAMPLE


,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty
0,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0
1,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
2,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
3,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0
4,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0
5,980531,Navan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0
6,980531,Sandyford,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
7,980532,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,1,IN_STOCK,0
8,980532,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,4,IN_STOCK,0
9,980532,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,2,IN_STOCK,0


### Step 1 — Stock Population & Integrity Validation

In [2]:
# ============================================================
# STOCK POPULATION & INTEGRITY VALIDATION
# ============================================================

required_cols = [
    "Model",
    "Store",
    "Category",
    "Description",
    "Quantity",
    "Stock_Status",
    "Outstanding_Order_Qty"
]

missing_cols = [
    col for col in required_cols
    if col not in fact_stock.columns
]

duplicate_model_store = fact_stock.duplicated(
    subset=["Model", "Store"]
).sum()

print("STOCK POPULATION & INTEGRITY")
print("=" * 75)

print(f"Stock records          : {len(fact_stock):,}")
print(f"Unique models          : {fact_stock['Model'].nunique():,}")
print(f"Stores                 : {fact_stock['Store'].nunique():,}")
print(f"Categories             : {fact_stock['Category'].nunique():,}")

print()
print(f"Total stock units      : {fact_stock['Quantity'].sum():,.0f}")
print(
    f"Outstanding order units: "
    f"{fact_stock['Outstanding_Order_Qty'].sum():,.0f}"
)

print("\nDATA INTEGRITY")
print("=" * 75)

print(f"Missing required columns : {missing_cols}")
print(f"Duplicate Model × Store  : {duplicate_model_store:,}")
print(
    f"Missing critical values  : "
    f"{fact_stock[required_cols].isna().sum().sum():,}"
)

print(
    f"Negative stock records   : "
    f"{(fact_stock['Quantity'] < 0).sum():,}"
)

print(
    f"Negative order records   : "
    f"{(fact_stock['Outstanding_Order_Qty'] < 0).sum():,}"
)

STOCK POPULATION & INTEGRITY
Stock records          : 2,681
Unique models          : 383
Stores                 : 7
Categories             : 27

Total stock units      : 6,001
Outstanding order units: 13

DATA INTEGRITY
Missing required columns : []
Duplicate Model × Store  : 0
Missing critical values  : 0
Negative stock records   : 12
Negative order records   : 0


### Step 2 — Inventory Position Analysis

In [3]:

stock_position = fact_stock.copy()

# ------------------------------------------------------------
# 1. Governed stock-position classification
# ------------------------------------------------------------

def classify_stock_position(qty):
    if qty < 0:
        return "NEGATIVE_STOCK"
    elif qty == 0:
        return "OUT_OF_STOCK"
    elif qty <= 2:
        return "LOW_STOCK"
    else:
        return "IN_STOCK"


stock_position["Inventory_Position"] = (
    stock_position["Quantity"]
    .apply(classify_stock_position)
)

# ------------------------------------------------------------
# 2. Position summary
# ------------------------------------------------------------

position_summary = (
    stock_position
    .groupby("Inventory_Position", as_index=False)
    .agg(
        Records=("Model", "size"),
        Models=("Model", "nunique"),
        Stock_Units=("Quantity", "sum")
    )
)

position_summary["Record_Share_%"] = (
    position_summary["Records"]
    / len(stock_position)
    * 100
)

position_order = [
    "IN_STOCK",
    "LOW_STOCK",
    "OUT_OF_STOCK",
    "NEGATIVE_STOCK"
]

position_summary["Inventory_Position"] = pd.Categorical(
    position_summary["Inventory_Position"],
    categories=position_order,
    ordered=True
)

position_summary = (
    position_summary
    .sort_values("Inventory_Position")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Display
# ------------------------------------------------------------

display_summary = position_summary.copy()

display_summary["Record_Share_%"] = (
    display_summary["Record_Share_%"]
    .map(lambda x: f"{x:.2f}%")
)

print("INVENTORY POSITION ANALYSIS")
print("=" * 75)

display(display_summary)

# ------------------------------------------------------------
# 4. Reconciliation
# ------------------------------------------------------------

print("\nINVENTORY RECONCILIATION")
print("=" * 75)

print(
    f"Stock records classified : "
    f"{position_summary['Records'].sum():,}"
)

print(
    f"Source stock records     : "
    f"{len(stock_position):,}"
)

print(
    "Population reconciles   :",
    position_summary["Records"].sum() == len(stock_position)
)

print(
    "Stock units reconcile   :",
    position_summary["Stock_Units"].sum()
    == stock_position["Quantity"].sum()
)

INVENTORY POSITION ANALYSIS


,Inventory_Position,Records,Models,Stock_Units,Record_Share_%
0,IN_STOCK,605,315,4320,22.57%
1,LOW_STOCK,1271,364,1694,47.41%
2,OUT_OF_STOCK,793,332,0,29.58%
3,NEGATIVE_STOCK,12,12,-13,0.45%



INVENTORY RECONCILIATION
Stock records classified : 2,681
Source stock records     : 2,681
Population reconciles   : True
Stock units reconcile   : True


### Step 3 — Store-Level Stock Distribution

In [4]:


store_stock = (
    stock_position
    .groupby("Store", as_index=False)
    .agg(
        Models=("Model", "nunique"),
        Net_Stock_Units=("Quantity", "sum"),
        Outstanding_Order_Qty=("Outstanding_Order_Qty", "sum"),
        Out_of_Stock_Positions=(
            "Inventory_Position",
            lambda x: (x == "OUT_OF_STOCK").sum()
        ),
        Low_Stock_Positions=(
            "Inventory_Position",
            lambda x: (x == "LOW_STOCK").sum()
        ),
        Negative_Stock_Positions=(
            "Inventory_Position",
            lambda x: (x == "NEGATIVE_STOCK").sum()
        )
    )
)

# Total model-store positions per store
store_records = (
    stock_position
    .groupby("Store")
    .size()
)

store_stock["Stock_Positions"] = (
    store_stock["Store"]
    .map(store_records)
)

# Availability rate
store_stock["Availability_Rate_%"] = (
    (
        store_stock["Stock_Positions"]
        - store_stock["Out_of_Stock_Positions"]
        - store_stock["Negative_Stock_Positions"]
    )
    / store_stock["Stock_Positions"]
    * 100
)

# Share of total governed stock
store_stock["Stock_Share_%"] = (
    store_stock["Net_Stock_Units"]
    / stock_position["Quantity"].sum()
    * 100
)

store_stock = (
    store_stock
    .sort_values("Net_Stock_Units", ascending=False)
    .reset_index(drop=True)
)

# Display formatting
display_store = store_stock.copy()

display_store["Availability_Rate_%"] = (
    display_store["Availability_Rate_%"]
    .map(lambda x: f"{x:.2f}%")
)

display_store["Stock_Share_%"] = (
    display_store["Stock_Share_%"]
    .map(lambda x: f"{x:.2f}%")
)

print("STORE-LEVEL STOCK DISTRIBUTION")
print("=" * 80)

display(display_store)

print("\nSTORE STOCK RECONCILIATION")
print("=" * 80)

print(f"Stores                  : {store_stock['Store'].nunique():,}")
print(f"Net stock units         : {store_stock['Net_Stock_Units'].sum():,.0f}")
print(
    "Stock units reconcile  :",
    store_stock["Net_Stock_Units"].sum()
    == stock_position["Quantity"].sum()
)

print(
    "Stock positions reconcile:",
    store_stock["Stock_Positions"].sum()
    == len(stock_position)
)

STORE-LEVEL STOCK DISTRIBUTION


,Store,Models,Net_Stock_Units,Outstanding_Order_Qty,Out_of_Stock_Positions,Low_Stock_Positions,Negative_Stock_Positions,Stock_Positions,Availability_Rate_%,Stock_Share_%
0,Gorey,383,2850,3,18,119,2,383,94.78%,47.49%
1,Cavan,383,1094,0,39,182,0,383,89.82%,18.23%
2,Blanch,383,871,3,48,203,3,383,86.68%,14.51%
3,Belfast,383,405,0,152,190,0,383,60.31%,6.75%
4,Dundrum,383,293,0,156,218,0,383,59.27%,4.88%
5,Sandyford,383,254,1,173,202,1,383,54.57%,4.23%
6,Navan,383,234,6,207,157,6,383,44.39%,3.90%



STORE STOCK RECONCILIATION
Stores                  : 7
Net stock units         : 6,001
Stock units reconcile  : True
Stock positions reconcile: True


### Step 4 — Model-Level Inventory Position

In [ ]:

model_stock = (
    stock_position
    .groupby(
        ["Model", "Category", "Description"],
        as_index=False
    )
    .agg(
        Total_Stock=("Quantity", "sum"),
        Outstanding_Order_Qty=("Outstanding_Order_Qty", "sum"),
        Stores_With_Stock=(
            "Quantity",
            lambda x: (x > 0).sum()
        ),
        Stores_Out_of_Stock=(
            "Quantity",
            lambda x: (x == 0).sum()
        ),
        Stores_Negative=(
            "Quantity",
            lambda x: (x < 0).sum()
        )
    )
)

# ------------------------------------------------------------
# Company-wide model stock status
# ------------------------------------------------------------

def classify_model_stock(qty):
    if qty < 0:
        return "NET_NEGATIVE_STOCK"
    elif qty == 0:
        return "NO_STOCK"
    elif qty <= 2:
        return "LOW_TOTAL_STOCK"
    else:
        return "STOCK_AVAILABLE"


model_stock["Model_Stock_Status"] = (
    model_stock["Total_Stock"]
    .apply(classify_model_stock)
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

model_summary = (
    model_stock
    .groupby("Model_Stock_Status", as_index=False)
    .agg(
        Models=("Model", "nunique"),
        Stock_Units=("Total_Stock", "sum"),
        Outstanding_Order_Qty=("Outstanding_Order_Qty", "sum")
    )
)

model_summary["Model_Share_%"] = (
    model_summary["Models"]
    / model_stock["Model"].nunique()
    * 100
)

display_summary = model_summary.copy()

display_summary["Model_Share_%"] = (
    display_summary["Model_Share_%"]
    .map(lambda x: f"{x:.2f}%")
)

print("MODEL-LEVEL INVENTORY POSITION")
print("=" * 75)

display(display_summary)

print("\nMODEL INVENTORY RECONCILIATION")
print("=" * 75)

print(f"Models                 : {model_stock['Model'].nunique():,}")
print(f"Net stock units        : {model_stock['Total_Stock'].sum():,.0f}")

print(
    "Model population valid:",
    model_stock["Model"].nunique()
    == fact_stock["Model"].nunique()
)

print(
    "Stock units reconcile:",
    model_stock["Total_Stock"].sum()
    == fact_stock["Quantity"].sum()
)

MODEL-LEVEL INVENTORY POSITION


,Model_Stock_Status,Models,Stock_Units,Outstanding_Order_Qty,Model_Share_%
0,STOCK_AVAILABLE,383,6001,13,100.00%



MODEL INVENTORY RECONCILIATION
Models                 : 383
Net stock units        : 6,001
Model population valid: True
Stock units reconcile: True


### Step 5 — Category-Level Inventory Distribution

In [ ]:
category_stock = (
    model_stock
    .groupby("Category", as_index=False)
    .agg(
        Models=("Model", "nunique"),
        Stock_Units=("Total_Stock", "sum"),
        Outstanding_Order_Qty=("Outstanding_Order_Qty", "sum")
    )
)

category_stock["Stock_Share_%"] = (
    category_stock["Stock_Units"]
    / model_stock["Total_Stock"].sum()
    * 100
)

category_stock["Avg_Stock_Per_Model"] = (
    category_stock["Stock_Units"]
    / category_stock["Models"]
)

category_stock = (
    category_stock
    .sort_values("Stock_Units", ascending=False)
    .reset_index(drop=True)
)

category_stock["Stock_Rank"] = (
    np.arange(1, len(category_stock) + 1)
)

display_category = category_stock[
    [
        "Stock_Rank",
        "Category",
        "Models",
        "Stock_Units",
        "Stock_Share_%",
        "Avg_Stock_Per_Model",
        "Outstanding_Order_Qty"
    ]
].copy()

display_category["Stock_Share_%"] = (
    display_category["Stock_Share_%"]
    .map(lambda x: f"{x:.2f}%")
)

display_category["Avg_Stock_Per_Model"] = (
    display_category["Avg_Stock_Per_Model"]
    .map(lambda x: f"{x:.2f}")
)

print("CATEGORY-LEVEL INVENTORY DISTRIBUTION")
print("=" * 80)

display(display_category)

print("\nCATEGORY RECONCILIATION")
print("=" * 80)

print(f"Categories            : {category_stock['Category'].nunique():,}")
print(f"Models                : {category_stock['Models'].sum():,}")
print(f"Stock units           : {category_stock['Stock_Units'].sum():,.0f}")

print(
    "Stock units reconcile:",
    category_stock["Stock_Units"].sum()
    == model_stock["Total_Stock"].sum()
)

CATEGORY-LEVEL INVENTORY DISTRIBUTION


,Stock_Rank,Category,Models,Stock_Units,Stock_Share_%,Avg_Stock_Per_Model,Outstanding_Order_Qty
0,1,SINGLE OVENS,59,916,15.26%,15.53,1
1,2,MICROWAVE OVENS,37,719,11.98%,19.43,2
2,3,WASHING MACHINES,34,678,11.30%,19.94,1
3,4,HOBS,36,567,9.45%,15.75,2
4,5,TUMBLE DRYERS,24,503,8.38%,20.96,1
5,6,INT DISHWASHERS,23,461,7.68%,20.04,1
6,7,INT FRIDGE FREEZERS,14,295,4.92%,21.07,0
7,8,INT MICROWAVES,13,193,3.22%,14.85,0
8,9,DOUBLE OVENS,12,168,2.80%,14.00,0
9,10,FRIDGE FREEZERS,15,168,2.80%,11.20,0



CATEGORY RECONCILIATION
Categories            : 27
Models                : 383
Stock units           : 6,001
Stock units reconcile: True


### Step 6 — Stock × Sales Integration

####  6.1 Stock ↔ Sales Model Matching Audit

Before evaluating inventory against demand, stock models are reconciled with the governed sales population.

This audit determines:

- how many inventory models have corresponding sales history,
- which stock models have no sales match,
- which sales products are absent from the current stock file,
- whether the model key provides a reliable basis for Stock × Sales integration.

No unmatched records are removed at this stage.

In [11]:
# ============================================================
# STEP 6.1 — STOCK ↔ SALES MODEL MATCHING AUDIT
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Load governed sales data
# ------------------------------------------------------------

sales_path = Path("../../data/processed/fact_sales.csv")

fact_sales = pd.read_csv(sales_path)

# ------------------------------------------------------------
# 2. Build unique matching populations
# ------------------------------------------------------------

stock_models = set(
    fact_stock["Model"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

sales_models = set(
    fact_sales["Product_Key"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

# ------------------------------------------------------------
# 3. Matching audit
# ------------------------------------------------------------

matched_models = stock_models.intersection(sales_models)

stock_only_models = stock_models.difference(sales_models)
sales_only_models = sales_models.difference(stock_models)

stock_match_rate = (
    len(matched_models) / len(stock_models) * 100
)

sales_match_rate = (
    len(matched_models) / len(sales_models) * 100
)

# ------------------------------------------------------------
# 4. Output
# ------------------------------------------------------------

print("STOCK ↔ SALES MODEL MATCHING AUDIT")
print("=" * 80)

print(f"Unique stock models        : {len(stock_models):,}")
print(f"Unique sales Product_Keys  : {len(sales_models):,}")
print(f"Matched models             : {len(matched_models):,}")

print()
print(f"Stock models without sales : {len(stock_only_models):,}")
print(f"Sales models without stock : {len(sales_only_models):,}")

print()
print(f"Stock model match rate     : {stock_match_rate:.2f}%")
print(f"Sales model match rate     : {sales_match_rate:.2f}%")

print("\nSAMPLE STOCK MODELS WITHOUT SALES MATCH")
print("=" * 80)
print(sorted(stock_only_models)[:20])

print("\nSAMPLE SALES MODELS WITHOUT STOCK MATCH")
print("=" * 80)
print(sorted(sales_only_models)[:20])

STOCK ↔ SALES MODEL MATCHING AUDIT
Unique stock models        : 383
Unique sales Product_Keys  : 767
Matched models             : 56

Stock models without sales : 327
Sales models without stock : 711

Stock model match rate     : 14.62%
Sales model match rate     : 7.30%

SAMPLE STOCK MODELS WITHOUT SALES MATCH
['980531', '980533', '980536', '980543', '980610', '980611', '980612', 'ABK818E6NC', 'B54CR71G0B', 'B58CT68H0B', 'B64CS71G0B', 'B64VS71G0B', 'B6ACH7AG7B', 'BFL553MB0B', 'BI710C1B1B', 'BRR29723EWW/EU', 'C1AMG84G1B', 'C1AMG84N1B', 'C24MR21G0B', 'C24MR21N0B']

SAMPLE SALES MODELS WITHOUT STOCK MATCH
['010-02384-10', '010-02784-00', '010-02784-01', '010-02839-00', '01950', '10009310', '10107860', '10234470', '102785', '103414675', '10795780', '110200', '1107129', '1107130', '112.000', '112.204', '112030', '112032', '112072', '113.004']


In [12]:
# ============================================================
# STEP 6.1A — LOAD PRODUCT DIMENSION FOR STOCK ↔ SALES BRIDGE
# ============================================================

from pathlib import Path
import pandas as pd

dim_product_path = Path("../../data/processed/dim_product.csv")

dim_product = pd.read_csv(dim_product_path)

print("DIM_PRODUCT BRIDGE CHECK")
print("=" * 80)

print(f"Rows    : {len(dim_product):,}")
print(f"Columns : {dim_product.shape[1]:,}")

print("\nCOLUMNS")
print("=" * 80)

for col in dim_product.columns:
    print(col)

DIM_PRODUCT BRIDGE CHECK
Rows    : 1,436
Columns : 18

COLUMNS
Product_ID
Product_Key
Product_Description
Product_Category
Record_Type
Source_Status
Source_Count
In_Sales
In_Stock
In_SOA
Sales_Stock_Code
Sales_Description
Sales_Category
Stock_Model
Stock_Description
Stock_Category
SOA_Model
SOA_Description


In [13]:
# ============================================================
# STEP 6.1B — GOVERNED STOCK → PRODUCT BRIDGE AUDIT
# ============================================================

# Normalise identifiers
stock_keys = (
    fact_stock["Model"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

bridge_keys = (
    dim_product.loc[dim_product["In_Stock"] == True, "Stock_Model"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

stock_models = set(stock_keys)
bridge_models = set(bridge_keys)

matched_models = stock_models & bridge_models
unmatched_models = stock_models - bridge_models

match_rate = (
    len(matched_models) / len(stock_models) * 100
    if stock_models else 0
)

print("GOVERNED STOCK → PRODUCT BRIDGE AUDIT")
print("=" * 80)

print(f"Unique stock models        : {len(stock_models):,}")
print(f"Stock models in dim_product: {len(bridge_models):,}")
print(f"Matched stock models       : {len(matched_models):,}")
print(f"Unmatched stock models     : {len(unmatched_models):,}")
print(f"Stock bridge match rate    : {match_rate:.2f}%")

print("\nSAMPLE UNMATCHED STOCK MODELS")
print("=" * 80)

print(sorted(unmatched_models)[:20])

GOVERNED STOCK → PRODUCT BRIDGE AUDIT
Unique stock models        : 383
Stock models in dim_product: 383
Matched stock models       : 383
Unmatched stock models     : 0
Stock bridge match rate    : 100.00%

SAMPLE UNMATCHED STOCK MODELS
[]


#### Step 6.2 — Build Stock × Sales Analytical Dataset

In [14]:


# ------------------------------------------------------------
# 1. Stock model → governed Product_ID bridge
# ------------------------------------------------------------

stock_bridge = (
    dim_product.loc[
        dim_product["In_Stock"] == True,
        ["Product_ID", "Stock_Model"]
    ]
    .dropna(subset=["Stock_Model"])
    .copy()
)

stock_bridge["Stock_Model"] = (
    stock_bridge["Stock_Model"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# 2. Prepare model-level stock
# ------------------------------------------------------------

stock_product = model_stock.copy()

stock_product["Model"] = (
    stock_product["Model"]
    .astype(str)
    .str.strip()
    .str.upper()
)

stock_product = stock_product.merge(
    stock_bridge,
    left_on="Model",
    right_on="Stock_Model",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 3. Aggregate governed sales by Product_ID
# ------------------------------------------------------------

sales_product = (
    fact_sales
    .groupby("Product_ID", as_index=False)
    .agg(
        Months_With_Sales=("Source_Month", "nunique"),
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum")
    )
)

sales_product["Units_Per_Month"] = np.where(
    sales_product["Months_With_Sales"] > 0,
    sales_product["Net_Units"] / sales_product["Months_With_Sales"],
    0
)


# ------------------------------------------------------------
# 4. Merge stock with sales
# ------------------------------------------------------------

stock_sales = stock_product.merge(
    sales_product,
    on="Product_ID",
    how="left",
    validate="one_to_one"
)

# Products with no sales history remain valid inventory products
stock_sales["Months_With_Sales"] = (
    stock_sales["Months_With_Sales"].fillna(0).astype(int)
)

stock_sales["Net_Units"] = stock_sales["Net_Units"].fillna(0)

stock_sales["Revenue"] = stock_sales["Revenue"].fillna(0)

stock_sales["Units_Per_Month"] = (
    stock_sales["Units_Per_Month"].fillna(0)
)


# ------------------------------------------------------------
# 5. Integration summary
# ------------------------------------------------------------

products_with_sales = (stock_sales["Months_With_Sales"] > 0).sum()
products_without_sales = (stock_sales["Months_With_Sales"] == 0).sum()

print("STOCK × SALES ANALYTICAL DATASET")
print("=" * 80)

print(f"Stock models              : {len(stock_sales):,}")
print(f"Unique Product_IDs        : {stock_sales['Product_ID'].nunique():,}")
print(f"Models with sales history : {products_with_sales:,}")
print(f"Models without sales      : {products_without_sales:,}")

print()
print(f"Total stock units         : {stock_sales['Total_Stock'].sum():,.0f}")
print(f"Net sales units           : {stock_sales['Net_Units'].sum():,.0f}")
print(f"Sales revenue             : £{stock_sales['Revenue'].sum():,.2f}")

print("\nINTEGRATION VALIDATION")
print("=" * 80)

print(
    "Product population valid:",
    len(stock_sales) == 383
    and stock_sales["Product_ID"].nunique() == 383
)

print(
    "Missing Product_IDs     :",
    stock_sales["Product_ID"].isna().sum()
)

print(
    "Stock units reconcile   :",
    stock_sales["Total_Stock"].sum()
    == model_stock["Total_Stock"].sum()
)

display(
    stock_sales[
        [
            "Product_ID",
            "Model",
            "Category",
            "Description",
            "Total_Stock",
            "Outstanding_Order_Qty",
            "Stores_With_Stock",
            "Months_With_Sales",
            "Net_Units",
            "Units_Per_Month",
            "Revenue"
        ]
    ].head(10)
)

STOCK × SALES ANALYTICAL DATASET
Stock models              : 383
Unique Product_IDs        : 383
Models with sales history : 56
Models without sales      : 327

Total stock units         : 6,001
Net sales units           : 75
Sales revenue             : £38,492.69

INTEGRATION VALIDATION
Product population valid: True
Missing Product_IDs     : 0
Stock units reconcile   : True


,Product_ID,Model,Category,Description,Total_Stock,Outstanding_Order_Qty,Stores_With_Stock,Months_With_Sales,Net_Units,Units_Per_Month,Revenue
0,221,980531,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,14,0,6,0,0.0,0.0,0.00
1,222,980532,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER MANUAL,17,0,6,1,1.0,1.0,63.74
2,223,980533,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT BLACK MANUAL,11,2,6,0,0.0,0.0,0.00
3,224,980535,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT SILVER,18,0,7,1,1.0,1.0,82.50
4,225,980536,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT BLACK,16,0,7,0,0.0,0.0,0.00
5,226,980538,MICROWAVE OVENS,DIMPLEX 23 LITRE 900 WATT SILVER,6,0,5,2,2.0,1.0,206.67
6,227,980539,MICROWAVE OVENS,Dimplex Black 23L 900 Watts Microwave,9,0,5,2,2.0,1.0,215.84
7,228,980543,MICROWAVE OVENS,Sona 20L 700W White Microwave,21,0,6,0,0.0,0.0,0.00
8,234,980585,MICROWAVE OVENS,Dimplex 26L 900W Combi Microwave with Grill,8,0,7,1,1.0,1.0,88.11
9,239,980610,MICROWAVE OVENS,Sona 20L 700W White Microwave,8,0,3,0,0.0,0.0,0.00


In [15]:
# ============================================================
# STEP 6.3 — STOCK × SALES OVERLAP VALIDATION
# ============================================================

bridge_check = dim_product[
    dim_product["In_Stock"] == True
].copy()

print("STOCK × SALES BRIDGE OVERLAP")
print("=" * 80)

print(
    "Stock products also marked In_Sales :",
    bridge_check["In_Sales"].sum()
)

print(
    "Stock products not marked In_Sales  :",
    (~bridge_check["In_Sales"]).sum()
)

print("\nSOURCE STATUS")
print("=" * 80)

print(
    bridge_check["Source_Status"]
    .value_counts(dropna=False)
)

STOCK × SALES BRIDGE OVERLAP
Stock products also marked In_Sales : 56
Stock products not marked In_Sales  : 327

SOURCE STATUS
Source_Status
STOCK_ONLY     319
SALES_STOCK     56
STOCK_SOA        8
Name: count, dtype: int64


### STEP 7 — STOCK COVER ANALYSIS

In [16]:

stock_cover = stock_sales[
    stock_sales["Months_With_Sales"] > 0
].copy()

# Only positive observed demand can produce meaningful stock cover
stock_cover["Stock_Cover_Months"] = np.where(
    stock_cover["Units_Per_Month"] > 0,
    stock_cover["Total_Stock"] / stock_cover["Units_Per_Month"],
    np.nan
)

valid_cover = stock_cover["Stock_Cover_Months"].notna()

print("STOCK COVER ANALYSIS")
print("=" * 80)

print(f"Sales-observed stock models : {len(stock_cover):,}")
print(f"Valid stock-cover models    : {valid_cover.sum():,}")
print(f"Non-positive demand models  : {(~valid_cover).sum():,}")

print("\nSTOCK COVER DISTRIBUTION")
print("=" * 80)

print(
    stock_cover.loc[
        valid_cover,
        "Stock_Cover_Months"
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    ).round(2)
)

print("\nVALIDATION")
print("=" * 80)

print(
    "Population reconciles:",
    valid_cover.sum() + (~valid_cover).sum()
    == len(stock_cover)
)

STOCK COVER ANALYSIS
Sales-observed stock models : 56
Valid stock-cover models    : 55
Non-positive demand models  : 1

STOCK COVER DISTRIBUTION
count    55.00
mean     14.16
std       8.67
min       3.00
25%       8.00
50%      13.00
75%      17.50
90%      21.60
max      54.00
Name: Stock_Cover_Months, dtype: float64

VALIDATION
Population reconciles: True


### Step 8 — Stock Cover Risk Segmentation

In [17]:

q1 = stock_cover.loc[
    stock_cover["Stock_Cover_Months"].notna(),
    "Stock_Cover_Months"
].quantile(0.25)

q3 = stock_cover.loc[
    stock_cover["Stock_Cover_Months"].notna(),
    "Stock_Cover_Months"
].quantile(0.75)


def classify_stock_cover(row):

    if pd.isna(row["Stock_Cover_Months"]):
        return "NO_POSITIVE_DEMAND"

    if row["Stock_Cover_Months"] < q1:
        return "LOWER_COVER"

    if row["Stock_Cover_Months"] > q3:
        return "HIGH_COVER"

    return "TYPICAL_COVER"


stock_cover["Stock_Cover_Segment"] = stock_cover.apply(
    classify_stock_cover,
    axis=1
)


cover_summary = (
    stock_cover
    .groupby("Stock_Cover_Segment", as_index=False)
    .agg(
        Models=("Product_ID", "nunique"),
        Stock_Units=("Total_Stock", "sum"),
        Net_Units=("Net_Units", "sum"),
        Avg_Cover_Months=("Stock_Cover_Months", "mean")
    )
)

print("STOCK COVER SEGMENTATION")
print("=" * 80)

print(f"Q1 threshold : {q1:.2f} months")
print(f"Q3 threshold : {q3:.2f} months")

display(
    cover_summary.round({
        "Avg_Cover_Months": 2
    })
)

print("\nVALIDATION")
print("=" * 80)

print(f"Models classified : {cover_summary['Models'].sum():,}")
print(f"Expected models   : {len(stock_cover):,}")

print(
    "Population reconciles:",
    cover_summary["Models"].sum() == len(stock_cover)
)

STOCK COVER SEGMENTATION
Q1 threshold : 8.00 months
Q3 threshold : 17.50 months


,Stock_Cover_Segment,Models,Stock_Units,Net_Units,Avg_Cover_Months
0,HIGH_COVER,14,388,16.0,24.93
1,LOWER_COVER,11,76,20.0,5.91
2,NO_POSITIVE_DEMAND,1,7,0.0,NaN
3,TYPICAL_COVER,30,400,39.0,12.16



VALIDATION
Models classified : 56
Expected models   : 56
Population reconciles: True


### Step 9 — Inventory Action Priorities

In [20]:
priority_models = stock_cover[
    stock_cover["Stock_Cover_Segment"].isin(
        ["HIGH_COVER", "NO_POSITIVE_DEMAND"]
    )
].copy()

# ------------------------------------------------------------
# 1. Assign inventory action
# ------------------------------------------------------------

priority_models["Inventory_Action"] = priority_models[
    "Stock_Cover_Segment"
].map({
    "HIGH_COVER": "REDUCE / REVIEW STOCK",
    "NO_POSITIVE_DEMAND": "DEMAND REVIEW"
})

# ------------------------------------------------------------
# 2. Select business fields
# ------------------------------------------------------------

priority_columns = [
    "Product_ID",
    "Model",
    "Category",
    "Description",
    "Total_Stock",
    "Outstanding_Order_Qty",
    "Net_Units",
    "Units_Per_Month",
    "Stock_Cover_Months",
    "Stock_Cover_Segment",
    "Inventory_Action"
]

inventory_priorities = (
    priority_models[priority_columns]
    .sort_values(
        ["Stock_Cover_Segment", "Stock_Cover_Months", "Total_Stock"],
        ascending=[True, False, False],
        na_position="last"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("INVENTORY ACTION PRIORITIES")
print("=" * 80)

print(
    "High-cover models          :",
    (inventory_priorities["Stock_Cover_Segment"] == "HIGH_COVER").sum()
)

print(
    "No-positive-demand models  :",
    (inventory_priorities["Stock_Cover_Segment"] == "NO_POSITIVE_DEMAND").sum()
)

print(
    "Priority models            :",
    len(inventory_priorities)
)

print(
    "Stock units under review   :",
    int(inventory_priorities["Total_Stock"].sum())
)

print(
    "Outstanding orders         :",
    int(inventory_priorities["Outstanding_Order_Qty"].sum())
)

print("\nPRIORITY MODEL DETAIL")
print("=" * 80)

display(inventory_priorities)

# ------------------------------------------------------------
# 4. Validation
# ------------------------------------------------------------

expected_priority_models = (
    (stock_cover["Stock_Cover_Segment"] == "HIGH_COVER").sum()
    +
    (stock_cover["Stock_Cover_Segment"] == "NO_POSITIVE_DEMAND").sum()
)

print("\nVALIDATION")
print("=" * 80)

print(f"Priority models identified : {len(inventory_priorities):,}")
print(f"Expected priority models   : {expected_priority_models:,}")
print(
    "Population reconciles     :",
    len(inventory_priorities) == expected_priority_models
)

INVENTORY ACTION PRIORITIES
High-cover models          : 14
No-positive-demand models  : 1
Priority models            : 15
Stock units under review   : 395
Outstanding orders         : 1

PRIORITY MODEL DETAIL


,Product_ID,Model,Category,Description,Total_Stock,Outstanding_Order_Qty,Net_Units,Units_Per_Month,Stock_Cover_Months,Stock_Cover_Segment,Inventory_Action
0,440,EDHI6285B,TUMBLE DRYERS,Electrolux 8kg Heat Pump Dryer,54,0,1.0,1.0,54.0,HIGH_COVER,REDUCE / REVIEW STOCK
1,448,EFI63142UD,WASHING MACHINES,Electrolux 10kg 1400 Spin Universal Dose Washer,35,0,1.0,1.0,35.0,HIGH_COVER,REDUCE / REVIEW STOCK
2,743,LIB60420C,HOBS,Electrolux 60cm Induction Hob,31,0,1.0,1.0,31.0,HIGH_COVER,REDUCE / REVIEW STOCK
3,1102,SI2641D,HOBS,Smeg 60cm 7.2kW Induction Hob,29,0,1.0,1.0,29.0,HIGH_COVER,REDUCE / REVIEW STOCK
4,758,LNT6NE18S,INT FRIDGE FREEZERS,Electrolux Built in Fridge Freezer 70:30,24,0,1.0,1.0,24.0,HIGH_COVER,REDUCE / REVIEW STOCK
5,760,LRR6436,HOBS,Electrolux 60cm Ceramic Touch Control Hob,22,0,1.0,1.0,22.0,HIGH_COVER,REDUCE / REVIEW STOCK
6,381,DI362DQ,INT DISHWASHERS,Smeg Integrated Dishwasher New,42,0,2.0,2.0,21.0,HIGH_COVER,REDUCE / REVIEW STOCK
7,1051,S187ZCX03G,INT DISHWASHERS,Neff N70 Integrated Dishwasher,21,1,1.0,1.0,21.0,HIGH_COVER,REDUCE / REVIEW STOCK
8,1234,TR848A4B2,TUMBLE DRYERS,AEG 8000 Series 8kg Heat Pump Dryer,20,0,1.0,1.0,20.0,HIGH_COVER,REDUCE / REVIEW STOCK
9,1287,VM131BL,INT MICROWAVES,CDA Black Integrated Microwave,20,0,1.0,1.0,20.0,HIGH_COVER,REDUCE / REVIEW STOCK



VALIDATION
Priority models identified : 15
Expected priority models   : 15
Population reconciles     : True


### Step 10 — Final Inventory Validation & Close

In [21]:
# ============================================================
# STEP 10 — FINAL INVENTORY ANALYSIS GATE
# ============================================================

print("FINAL INVENTORY ANALYSIS GATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Core population checks
# ------------------------------------------------------------

stock_records = len(fact_stock)
unique_models = fact_stock["Model"].nunique()
stores = fact_stock["Store"].nunique()
categories = fact_stock["Category"].nunique()

# ------------------------------------------------------------
# 2. Stock reconciliation
# ------------------------------------------------------------

stock_units = fact_stock["Quantity"].sum()
model_stock_units = model_stock["Total_Stock"].sum()

# ------------------------------------------------------------
# 3. Stock ↔ Sales bridge
# ------------------------------------------------------------

bridge_match_rate = (
    dim_product.loc[dim_product["In_Stock"] == True, "Stock_Model"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .nunique()
)

# ------------------------------------------------------------
# 4. Demand-linked population
# ------------------------------------------------------------

sales_observed_models = len(stock_cover)
valid_cover_models = stock_cover["Stock_Cover_Months"].notna().sum()

# ------------------------------------------------------------
# 5. Priority population
# ------------------------------------------------------------

priority_count = len(inventory_priorities)

# ------------------------------------------------------------
# 6. Gate conditions
# ------------------------------------------------------------

population_pass = (
    stock_records == 2681
    and unique_models == 383
    and stores == 7
    and categories == 27
)

stock_reconciliation_pass = (
    stock_units == 6001
    and model_stock_units == stock_units
)

bridge_pass = (
    bridge_match_rate == 383
)

cover_pass = (
    sales_observed_models == 56
    and valid_cover_models == 55
)

priority_pass = (
    priority_count == 15
)

# ------------------------------------------------------------
# 7. Output
# ------------------------------------------------------------

print(f"Stock records              : {stock_records:,}")
print(f"Unique models              : {unique_models:,}")
print(f"Stores                     : {stores:,}")
print(f"Categories                 : {categories:,}")
print(f"Net stock units            : {stock_units:,.0f}")

print()
print(f"Sales-observed stock models: {sales_observed_models:,}")
print(f"Valid stock-cover models   : {valid_cover_models:,}")
print(f"Priority models            : {priority_count:,}")

print("\nGATE CONDITIONS")
print("=" * 80)

print("Population integrity       :", population_pass)
print("Stock reconciliation       :", stock_reconciliation_pass)
print("Product bridge integrity   :", bridge_pass)
print("Stock-cover validity       :", cover_pass)
print("Priority reconciliation    :", priority_pass)

final_inventory_pass = all([
    population_pass,
    stock_reconciliation_pass,
    bridge_pass,
    cover_pass,
    priority_pass
])

print("\nFINAL INVENTORY GATE")
print("=" * 80)

print(
    "STATUS:",
    "PASS" if final_inventory_pass else "FAIL"
)

FINAL INVENTORY ANALYSIS GATE
Stock records              : 2,681
Unique models              : 383
Stores                     : 7
Categories                 : 27
Net stock units            : 6,001

Sales-observed stock models: 56
Valid stock-cover models   : 55
Priority models            : 15

GATE CONDITIONS
Population integrity       : True
Stock reconciliation       : True
Product bridge integrity   : True
Stock-cover validity       : True
Priority reconciliation    : True

FINAL INVENTORY GATE
STATUS: PASS


# Inventory / Stock Analysis — Key Findings

## 1. Inventory Population

- The governed stock dataset contains **2,681 stock records**.
- Inventory covers **383 unique product models** across **7 stores** and **27 product categories**.
- Total net stock holding is **6,001 units**.
- Only **13 units** are currently recorded as outstanding orders.
- No duplicate `Model × Store` combinations or missing critical inventory fields were identified.

---

## 2. Inventory Position

Across the 2,681 store-model inventory positions:

- **605 positions (22.57%)** are classified as `IN_STOCK`.
- **1,271 positions (47.41%)** are classified as `LOW_STOCK`.
- **793 positions (29.58%)** are `OUT_OF_STOCK`.
- **12 positions (0.45%)** contain negative stock.

This indicates that a substantial proportion of store-level inventory positions are either low-stock or out-of-stock, while negative stock records represent a small but identifiable data/operational exception requiring review.

---

## 3. Store-Level Inventory Distribution

Inventory is unevenly distributed across the seven stores.

- **Gorey** holds the largest stock position with **2,850 units**.
- **Cavan** holds **1,094 units**.
- **Blanch** holds **871 units**.
- Belfast, Dundrum, Sandyford and Navan hold materially lower stock positions.
- **Navan** has the highest number of out-of-stock positions (**207**) and also the highest number of negative-stock positions (**6**).

The store-level results indicate potential opportunities for inventory balancing and stock reallocation between locations.

---

## 4. Category-Level Stock Concentration

Inventory is concentrated primarily in major appliance categories.

The largest stock-holding categories are:

| Rank | Category | Stock Units | Stock Share |
|---|---|---:|---:|
| 1 | SINGLE OVENS | 916 | 15.26% |
| 2 | MICROWAVE OVENS | 719 | 11.98% |
| 3 | WASHING MACHINES | 678 | 11.30% |
| 4 | HOBS | 567 | 9.45% |
| 5 | TUMBLE DRYERS | 503 | 8.38% |

These five categories collectively represent a substantial proportion of the company's total inventory exposure.

---

## 5. Stock × Sales Product Integration

The governed `dim_product` bridge was used to connect inventory models with the sales product population.

- Stock population: **383 models**
- Models with observed sales history: **56**
- Models without observed sales history: **327**
- The stock population remained fully reconciled after integration.

Therefore, demand-based inventory metrics such as stock cover are calculated only for the **56 stock models with observable sales history**.

This limitation is explicitly preserved rather than assigning artificial demand to products without matched sales history.

---

## 6. Stock Cover

Of the **56 sales-observed stock models**:

- **55 models** have valid positive demand and therefore calculable stock cover.
- **1 model** has no positive demand.
- Median stock cover is approximately **13 months**.
- Mean stock cover is approximately **14.16 months**.
- The observed range extends from **3 months to 54 months**.

Using the observed stock-cover distribution:

- `LOWER_COVER`: **11 models**, averaging **5.91 months**
- `TYPICAL_COVER`: **30 models**, averaging **12.16 months**
- `HIGH_COVER`: **14 models**, averaging **24.93 months**
- `NO_POSITIVE_DEMAND`: **1 model**

The high-cover population represents the clearest inventory reduction/review opportunity within the demand-observed product population.

---

## 7. Inventory Action Priorities

The analysis identified **15 priority models** requiring commercial inventory review:

- **14 HIGH_COVER models**
- **1 NO_POSITIVE_DEMAND model**
- **395 stock units** are held across these priority models.
- **1 outstanding order unit** is associated with the priority population.

Several high-cover products have particularly large estimated cover periods, including approximately **54, 35, 31, 29, 24, 22 and 21 months**.

These models should be reviewed for possible:

- stock reduction,
- inter-store redistribution,
- purchasing restraint,
- promotional activity,
- pricing intervention,
- or demand validation.

These classifications are decision-support signals rather than automatic replenishment or delisting decisions.

---

## 8. Analytical Limitation

Stock-cover analysis is currently limited to **56 of the 383 stock models** because only these models have matched observable sales history through the governed product bridge.

Therefore, stock-cover findings must **not be generalized to the complete 383-model inventory population**.

The remaining **327 models** should retain their inventory positions without inferred demand until sufficient sales linkage or additional demand history becomes available.

---

## Final Validation

The final inventory analysis gate passed successfully:

- Population integrity: **PASS**
- Stock reconciliation: **PASS**
- Product bridge integrity: **PASS**
- Stock-cover validity: **PASS**
- Priority reconciliation: **PASS**

### Final Status: **PASS**

The Inventory / Stock Analysis notebook is complete and provides a governed analytical foundation for inventory monitoring, stock balancing, demand-based stock-cover assessment, and future inventory optimisation.